In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Libraries imported successfully!")

PyTorch version: 2.14.0+cpu
NumPy version: 2.2.6
Libraries imported successfully!


In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/tickets.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Dataset loaded successfully!
Rows: 500
Columns: 12


,ticket_id,employee_name,department,employee_type,subject,description,category,priority,status,created_at,resolved_at,resolution_time_hours
0,1,Employee_37,Sales,Permanent,Internet connection is very slow,Internet connection is very slow,Network,High,Resolved,2026-09-05 06:13:51,2026-09-05 17:06:39,10.88
1,2,Employee_47,Finance,Intern,Software is showing an error,Software is showing an error,Software,Medium,Open,2026-04-25 12:29:51,NaN,27.75
2,3,Employee_48,HR,Intern,Mouse is not responding,Mouse is not responding,Hardware,Low,Resolved,2026-07-03 08:02:51,2026-07-04 06:50:51,22.80
3,4,Employee_41,Finance,Contract,Application is crashing,Application is crashing,Software,Medium,In Progress,2026-05-09 20:05:51,NaN,22.59
4,5,Employee_86,Finance,Intern,Unable to send email,Unable to send email,Email,High,Resolved,2026-08-23 19:18:51,2026-08-23 20:29:03,1.17


In [4]:
X = df["description"]
y = df["category"]

print("Input variable (X):", X.name)
print("Target variable (y):", y.name)

print("\nNumber of samples:", len(X))
print("\nCategories:")
print(y.value_counts())

Input variable (X): description
Target variable (y): category

Number of samples: 500

Categories:
category
Account     93
Network     92
Email       81
Software    79
Security    79
Hardware    76
Name: count, dtype: int64


In [5]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Category encoding completed!")

print("\nCategory mapping:")
for category, number in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"{category} -> {number}")

print("\nFirst 10 encoded values:")
print(y_encoded[:10])

Category encoding completed!

Category mapping:
Account -> 0
Email -> 1
Hardware -> 2
Network -> 3
Security -> 4
Software -> 5

First 10 encoded values:
[3 5 2 5 1 2 1 5 3 4]


In [6]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

tokenized_texts = X.apply(tokenize)

print("Text cleaning and tokenization completed!")

print("\nOriginal text:")
print(X.iloc[0])

print("\nTokenized text:")
print(tokenized_texts.iloc[0])

Text cleaning and tokenization completed!

Original text:
Internet connection is very slow

Tokenized text:
['internet', 'connection', 'is', 'very', 'slow']


In [7]:
from collections import Counter

word_counts = Counter()

for tokens in tokenized_texts:
    word_counts.update(tokens)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in word_counts.items():
    vocab[word] = len(vocab)

print("Vocabulary created successfully!")
print("Vocabulary size:", len(vocab))

print("\nFirst 20 vocabulary words:")
print(list(vocab.items())[:20])

Vocabulary created successfully!
Vocabulary size: 66

First 20 vocabulary words:
[('<PAD>', 0), ('<UNK>', 1), ('internet', 2), ('connection', 3), ('is', 4), ('very', 5), ('slow', 6), ('software', 7), ('showing', 8), ('an', 9), ('error', 10), ('mouse', 11), ('not', 12), ('responding', 13), ('application', 14), ('crashing', 15), ('unable', 16), ('to', 17), ('send', 18), ('email', 19)]


In [8]:
sequences = []

for tokens in tokenized_texts:
    sequence = [vocab.get(word, vocab["<UNK>"]) for word in tokens]
    sequences.append(sequence)

print("Text converted into numerical sequences!")

print("\nOriginal tokens:")
print(tokenized_texts.iloc[0])

print("\nNumerical sequence:")
print(sequences[0])

Text converted into numerical sequences!

Original tokens:
['internet', 'connection', 'is', 'very', 'slow']

Numerical sequence:
[2, 3, 4, 5, 6]


In [9]:
from torch.nn.utils.rnn import pad_sequence

sequence_tensors = [
    torch.tensor(sequence, dtype=torch.long)
    for sequence in sequences
]

padded_sequences = pad_sequence(
    sequence_tensors,
    batch_first=True,
    padding_value=vocab["<PAD>"]
)

print("Sequence padding completed!")
print("Padded data shape:", padded_sequences.shape)

print("\nFirst padded sequence:")
print(padded_sequences[0])

Sequence padding completed!
Padded data shape: torch.Size([500, 5])

First padded sequence:
tensor([2, 3, 4, 5, 6])


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    torch.tensor(y_encoded, dtype=torch.long),
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Data split completed!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training sequence shape:", X_train.shape)
print("Testing sequence shape:", X_test.shape)

Data split completed!
Training samples: 400
Testing samples: 100
Training sequence shape: torch.Size([400, 5])
Testing sequence shape: torch.Size([100, 5])


In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("PyTorch DataLoaders created successfully!")
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

PyTorch DataLoaders created successfully!
Training batches: 13
Testing batches: 4


In [12]:
class TicketRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(TicketRNN, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=vocab["<PAD>"]
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)

        output, hidden = self.rnn(embedded)

        last_hidden = hidden[-1]

        output = self.fc(last_hidden)

        return output


vocab_size = len(vocab)
embedding_dim = 64
hidden_dim = 64
num_classes = len(label_encoder.classes_)

model = TicketRNN(
    vocab_size,
    embedding_dim,
    hidden_dim,
    num_classes
)

print("RNN model created successfully!")
print(model)

RNN model created successfully!
TicketRNN(
  (embedding): Embedding(66, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=6, bias=True)
)


In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function and optimizer created successfully!")
print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")

Loss function and optimizer created successfully!
Loss function: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.001


In [14]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    total_loss = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"Loss: {average_loss:.4f}"
    )

print("\nRNN training completed!")

Epoch [1/10], Loss: 1.5865
Epoch [2/10], Loss: 1.1560
Epoch [3/10], Loss: 0.7760
Epoch [4/10], Loss: 0.4620
Epoch [5/10], Loss: 0.2723
Epoch [6/10], Loss: 0.1664
Epoch [7/10], Loss: 0.0997
Epoch [8/10], Loss: 0.0626
Epoch [9/10], Loss: 0.0435
Epoch [10/10], Loss: 0.0325

RNN training completed!


In [15]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.numpy())
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_predictions)

print("RNN Accuracy:", accuracy)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=label_encoder.classes_
    )
)

RNN Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

     Account       1.00      1.00      1.00        19
       Email       1.00      1.00      1.00        16
    Hardware       1.00      1.00      1.00        15
     Network       1.00      1.00      1.00        18
    Security       1.00      1.00      1.00        16
    Software       1.00      1.00      1.00        16

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [16]:
new_tickets = [
    "My VPN is not connecting",
    "My laptop keyboard is not working",
    "I cannot login to my account",
    "I received a suspicious email"
]

new_tokenized = [tokenize(ticket) for ticket in new_tickets]

new_sequences = [
    [vocab.get(word, vocab["<UNK>"]) for word in tokens]
    for tokens in new_tokenized
]

new_tensors = [
    torch.tensor(sequence, dtype=torch.long)
    for sequence in new_sequences
]

new_padded = pad_sequence(
    new_tensors,
    batch_first=True,
    padding_value=vocab["<PAD>"]
)

model.eval()

with torch.no_grad():
    outputs = model(new_padded)
    predictions = torch.argmax(outputs, dim=1)

predicted_categories = label_encoder.inverse_transform(
    predictions.numpy()
)

print("RNN Predictions:\n")

for ticket, category in zip(new_tickets, predicted_categories):
    print(f"Ticket: {ticket}")
    print(f"Predicted Category: {category}")
    print()

RNN Predictions:

Ticket: My VPN is not connecting
Predicted Category: Network

Ticket: My laptop keyboard is not working
Predicted Category: Email

Ticket: I cannot login to my account
Predicted Category: Security

Ticket: I received a suspicious email
Predicted Category: Email

